# XLK baseline: market information and news sentiment

## tl;dr

This notebook tests whether daily RavenPack news sentiment adds predictive value for the next trading day's direction of the Technology Select Sector SPDR Fund (XLK).

Current run: the market-only model averages AUC-ROC 0.527 and directional accuracy 53.6%; adding news averages AUC-ROC 0.514 and accuracy 53.8%; the always-up reference reaches 55.3% accuracy. The design is a descriptive research baseline, not an investment strategy or recommendation.

## Context & Methods

### Research question

Does news sentiment improve prediction of whether XLK will have a positive return during the next CRSP trading session, beyond recent XLK returns, volatility, and volume?

### Key assumptions

- The target is `fwd_1d_positive`, already calculated from the next available XLK trading session in the CRSP gold table.
- News is aggregated by calendar date and joined to each market session on `session_date`. The model uses only information available in the current row and prior market sessions; future holdout rows are never used for training or scaling.
- All rows are pooled into one XLK classifier. The experiment evaluates predictive ranking and classification, not causal impact or trading profitability.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

# Allow execution from either the repository root or its data_collection folder.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'data_collection').exists():
    if (REPO_ROOT.parent / 'data_collection').exists():
        REPO_ROOT = REPO_ROOT.parent
    else:
        raise FileNotFoundError('Run this notebook from the repository root or data_collection/.')

DATA_DIR = REPO_ROOT / 'data_collection'
OUTPUT_DIR = REPO_ROOT / 'model_outputs'
MARKET_PATH = DATA_DIR / 'market_daily_df.csv'
NEWS_PATH = DATA_DIR / 'news_daily_df.csv'
METRICS_PATH = OUTPUT_DIR / 'xlk_walk_forward_metrics.csv'
PREDICTIONS_PATH = OUTPUT_DIR / 'xlk_holdout_predictions.csv'
FIGURE_PATH = OUTPUT_DIR / 'xlk_baseline_overview.png'

TARGET_TICKER = 'XLK'
START_DATE = pd.Timestamp('2020-01-01')
END_DATE = pd.Timestamp('2025-12-31')
TEST_YEARS = [2022, 2023, 2024, 2025]

BASELINE_FEATURES = [
    'return_lag_1',
    'return_lag_5',
    'return_lag_20',
    'return_mean_5',
    'return_mean_20',
    'return_std_20',
    'volume_log_lag_1',
]
SENTIMENT_FEATURES = [
    'mean_event_sentiment_score',
    'positive_event_share',
    'negative_event_share',
    'news_volume_log',
    'source_count_log',
]

print(f'Project folder: {REPO_ROOT}')
print(f'XLK market data: {MARKET_PATH}')
print(f'RavenPack daily news: {NEWS_PATH}')
print(f'Holdout years: {TEST_YEARS}')
print(f'Model outputs: {OUTPUT_DIR}')

## Data

### 1. Load and validate the XLK market panel

The market file is the engineered gold table created by `data_collection/02_crsp_sector_etf_price_extraction.ipynb`. Filtering to XLK keeps the same CRSP price, return, volume, and forward-label definitions used by the sector experiment.

In [ ]:
required_market_columns = {
    'session_date', 'ticker', 'daily_return', 'price', 'volume',
    'fwd_1d_return', 'fwd_1d_positive'
}
required_news_columns = {
    'session_date', 'event_record_count', 'unique_source_count',
    'mean_event_sentiment_score', 'positive_event_share', 'negative_event_share'
}

market_all = pd.read_csv(MARKET_PATH, parse_dates=['session_date'])
market_missing = required_market_columns - set(market_all.columns)
if market_missing:
    raise ValueError(f'Market table is missing required columns: {sorted(market_missing)}')

market = market_all.loc[
    market_all['ticker'].eq(TARGET_TICKER)
    & market_all['session_date'].between(START_DATE, END_DATE)
].copy()
if market.empty:
    raise ValueError(f'No {TARGET_TICKER} rows were found in the requested date range.')

duplicate_market_rows = int(market.duplicated(['ticker', 'session_date']).sum())
assert duplicate_market_rows == 0, f'Duplicate {TARGET_TICKER} sessions: {duplicate_market_rows}'
market = market.sort_values(['ticker', 'session_date']).reset_index(drop=True)

# Spot-check the engineered forward return against the next XLK session.
market['next_daily_return'] = market.groupby('ticker')['daily_return'].shift(-1)
return_check = market[['fwd_1d_return', 'next_daily_return']].dropna()
assert np.allclose(return_check['fwd_1d_return'], return_check['next_daily_return'], atol=1e-12)
market = market.drop(columns='next_daily_return')

print(f'XLK rows: {len(market):,}')
print(f'XLK coverage: {market["session_date"].min().date()} to {market["session_date"].max().date()}')
print(f'XLK positive next-session labels: {market["fwd_1d_positive"].mean():.1%} of rows with a known outcome')
display(market[['session_date', 'ticker', 'daily_return', 'price', 'volume', 'fwd_1d_positive']].head())

In [ ]:
news = pd.read_csv(NEWS_PATH, parse_dates=['session_date'])
news_missing = required_news_columns - set(news.columns)
if news_missing:
    raise ValueError(f'News table is missing required columns: {sorted(news_missing)}')

news = news.loc[news['session_date'].between(START_DATE, END_DATE)].copy()
duplicate_news_dates = int(news.duplicated('session_date').sum())
news = news.drop_duplicates('session_date').sort_values('session_date')

panel = market.merge(news, on='session_date', how='left', validate='many_to_one')
panel = panel.sort_values(['ticker', 'session_date']).reset_index(drop=True)

grouped_returns = panel.groupby('ticker')['daily_return']
for lag in [1, 5, 20]:
    panel[f'return_lag_{lag}'] = grouped_returns.shift(lag)
panel['return_mean_5'] = grouped_returns.transform(
    lambda values: values.shift(1).rolling(5, min_periods=5).mean()
)
panel['return_mean_20'] = grouped_returns.transform(
    lambda values: values.shift(1).rolling(20, min_periods=20).mean()
)
panel['return_std_20'] = grouped_returns.transform(
    lambda values: values.shift(1).rolling(20, min_periods=20).std(ddof=0)
)

panel['volume_log'] = np.log1p(panel['volume'].clip(lower=0))
panel['volume_log_lag_1'] = panel.groupby('ticker')['volume_log'].shift(1)
panel['news_volume_log'] = np.log1p(panel['event_record_count'].clip(lower=0))
panel['source_count_log'] = np.log1p(panel['unique_source_count'].clip(lower=0))

required_model_columns = BASELINE_FEATURES + SENTIMENT_FEATURES + ['fwd_1d_positive']
panel = panel.dropna(subset=required_model_columns).copy()
panel['target'] = panel['fwd_1d_positive'].astype(int)

print(f'News dates loaded: {news["session_date"].nunique():,}')
print(f'Duplicate news dates removed before merge: {duplicate_news_dates:,}')
print(f'Rows ready for modeling after feature history and join checks: {len(panel):,}')
print(f'Modeling coverage: {panel["session_date"].min().date()} to {panel["session_date"].max().date()}')
display(panel[['session_date', 'ticker', 'target'] + BASELINE_FEATURES[:3] + SENTIMENT_FEATURES[:2]].head())

### 2. Define the evaluation functions

The optimizer is a small regularized logistic regression implemented directly in NumPy. Features are standardized using training data only. AUC measures ranking quality; F1 and directional accuracy use a 0.50 probability threshold.

In [ ]:
def sigmoid(values: np.ndarray) -> np.ndarray:
    clipped = np.clip(values, -35.0, 35.0)
    return 1.0 / (1.0 + np.exp(-clipped))


def fit_logistic(
    x_train: np.ndarray,
    y_train: np.ndarray,
    learning_rate: float = 0.08,
    iterations: int = 2500,
    l2: float = 0.05,
) -> tuple[np.ndarray, float]:
    weights = np.zeros(x_train.shape[1], dtype=float)
    intercept = 0.0
    n_rows = float(len(y_train))
    for _ in range(iterations):
        probabilities = sigmoid(x_train @ weights + intercept)
        residual = probabilities - y_train
        weights -= learning_rate * ((x_train.T @ residual) / n_rows + l2 * weights)
        intercept -= learning_rate * residual.mean()
    return weights, intercept


def standardize(x_train: pd.DataFrame, x_test: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    train_mean = x_train.mean(axis=0)
    train_std = x_train.std(axis=0, ddof=0).replace(0.0, 1.0)
    train_scaled = ((x_train - train_mean) / train_std).to_numpy(dtype=float)
    test_scaled = ((x_test - train_mean) / train_std).to_numpy(dtype=float)
    return train_scaled, test_scaled


def roc_auc(y_true: np.ndarray, scores: np.ndarray) -> float:
    positives = y_true == 1
    negatives = y_true == 0
    n_positive = int(positives.sum())
    n_negative = int(negatives.sum())
    if n_positive == 0 or n_negative == 0:
        return float('nan')
    order = np.argsort(scores, kind='mergesort')
    sorted_scores = scores[order]
    ranks = np.empty(len(scores), dtype=float)
    start = 0
    while start < len(scores):
        end = start + 1
        while end < len(scores) and sorted_scores[end] == sorted_scores[start]:
            end += 1
        ranks[order[start:end]] = (start + 1 + end) / 2.0
        start = end
    positive_rank_sum = ranks[positives].sum()
    return float((positive_rank_sum - n_positive * (n_positive + 1) / 2.0) / (n_positive * n_negative))


def classification_metrics(y_true: np.ndarray, probabilities: np.ndarray) -> dict[str, float]:
    predictions = (probabilities >= 0.5).astype(int)
    true_positive = int(((predictions == 1) & (y_true == 1)).sum())
    false_positive = int(((predictions == 1) & (y_true == 0)).sum())
    false_negative = int(((predictions == 0) & (y_true == 1)).sum())
    denominator = 2 * true_positive + false_positive + false_negative
    f1 = 0.0 if denominator == 0 else 2.0 * true_positive / denominator
    return {
        'auc_roc': roc_auc(y_true, probabilities),
        'f1': float(f1),
        'directional_accuracy': float((predictions == y_true).mean()),
    }

## Results

### 3. Run the chronological holdout experiment

Each test year is evaluated only after training on earlier dates. The benchmark is deliberately simple: it always predicts a positive next-session return.

In [ ]:
model_features = {
    'Market-only model': BASELINE_FEATURES,
    'Market + news sentiment model': BASELINE_FEATURES + SENTIMENT_FEATURES,
}
metric_rows = []
prediction_rows = []

for test_year in TEST_YEARS:
    train_mask = panel['session_date'] < pd.Timestamp(f'{test_year}-01-01')
    test_mask = panel['session_date'].dt.year.eq(test_year)
    train = panel.loc[train_mask]
    test = panel.loc[test_mask]
    if train.empty or test.empty:
        print(f'Skipping {test_year}: train rows={len(train)}, test rows={len(test)}')
        continue

    y_train = train['target'].to_numpy(dtype=int)
    y_test = test['target'].to_numpy(dtype=int)
    print(f'{test_year}: train={len(train):,}, test={len(test):,}, test positive={y_test.mean():.1%}')

    for model_name, feature_columns in model_features.items():
        x_train, x_test = standardize(train[feature_columns], test[feature_columns])
        weights, intercept = fit_logistic(x_train, y_train)
        probabilities = sigmoid(x_test @ weights + intercept)
        metrics = classification_metrics(y_test, probabilities)

        for metric, score in metrics.items():
            metric_rows.append({
                'ticker': TARGET_TICKER,
                'model': model_name,
                'fold': test_year,
                'metric': metric,
                'score': score,
            })

        predictions = (probabilities >= 0.5).astype(int)
        for row, probability, prediction in zip(test.itertuples(index=False), probabilities, predictions):
            prediction_rows.append({
                'ticker': TARGET_TICKER,
                'model': model_name,
                'fold': test_year,
                'session_date': row.session_date.date().isoformat(),
                'y_true': int(row.target),
                'y_pred': int(prediction),
                'y_prob': float(probability),
            })

    always_up_probabilities = np.ones(len(test), dtype=float)
    always_up_metrics = classification_metrics(y_test, always_up_probabilities)
    for metric, score in always_up_metrics.items():
        metric_rows.append({
            'ticker': TARGET_TICKER,
            'model': 'Always-up reference',
            'fold': test_year,
            'metric': metric,
            'score': score,
        })
    for row in test.itertuples(index=False):
        prediction_rows.append({
            'ticker': TARGET_TICKER,
            'model': 'Always-up reference',
            'fold': test_year,
            'session_date': row.session_date.date().isoformat(),
            'y_true': int(row.target),
            'y_pred': 1,
            'y_prob': 1.0,
        })

metrics_df = pd.DataFrame(metric_rows)
predictions_df = pd.DataFrame(prediction_rows)
if metrics_df.empty or predictions_df.empty:
    raise ValueError('The walk-forward run produced no evaluation rows.')

OUTPUT_DIR.mkdir(exist_ok=True)
metrics_df.to_csv(METRICS_PATH, index=False)
predictions_df.to_csv(PREDICTIONS_PATH, index=False)

summary_df = (
    metrics_df.groupby(['model', 'metric'], as_index=False)['score']
    .agg(mean='mean', std='std', folds='count')
    .sort_values(['metric', 'model'])
)
print(f'Wrote {METRICS_PATH.relative_to(REPO_ROOT)}')
print(f'Wrote {PREDICTIONS_PATH.relative_to(REPO_ROOT)}')
display(summary_df.style.format({'mean': '{:.4f}', 'std': '{:.4f}'}))

In [ ]:
# Chart contract: an explanatory flow plus a same-scale model comparison.
# The bar chart uses zero as its baseline and labels the four-year holdout scope.
fig = plt.figure(figsize=(15, 6))
layout = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.35])

flow_axis = fig.add_subplot(layout[0, 0])
flow_axis.set_xlim(0, 1)
flow_axis.set_ylim(0, 1)
flow_axis.axis('off')
flow_axis.set_title('XLK model structure', fontsize=12, fontweight='bold', color='#222222')

def add_box(axis, x, y, width, height, label, color):
    box = FancyBboxPatch(
        (x, y), width, height, boxstyle='round,pad=0.02,rounding_size=0.03',
        facecolor=color, edgecolor='#555555', linewidth=1,
    )
    axis.add_patch(box)
    axis.text(x + width / 2, y + height / 2, label, ha='center', va='center', fontsize=9, color='#222222')

add_box(flow_axis, 0.02, 0.63, 0.28, 0.18, 'Market features\nreturns, volatility, volume', '#d9eaf7')
add_box(flow_axis, 0.02, 0.28, 0.28, 0.18, 'News features\ndaily RavenPack sentiment', '#e3f2df')
add_box(flow_axis, 0.40, 0.46, 0.25, 0.22, 'Regularized\nlogistic regression', '#f6e5c3')
add_box(flow_axis, 0.75, 0.46, 0.23, 0.22, 'Next session\nXLK up / down', '#eadcf5')
for start, end in [((0.30, 0.72), (0.40, 0.58)), ((0.30, 0.37), (0.40, 0.56)), ((0.65, 0.57), (0.75, 0.57))]:
    flow_axis.add_patch(FancyArrowPatch(start, end, arrowstyle='-|>', mutation_scale=14, linewidth=1.5, color='#666666'))

metric_axis = fig.add_subplot(layout[0, 1])
metric_plot = summary_df.pivot(index='model', columns='metric', values='mean')
metric_plot = metric_plot[['auc_roc', 'directional_accuracy', 'f1']].rename(
    columns={'auc_roc': 'AUC-ROC', 'directional_accuracy': 'Accuracy', 'f1': 'F1 score'}
)
metric_plot.plot(kind='bar', ax=metric_axis, width=0.78, color=['#4c78a8', '#b9770e', '#8a8a8a'])
metric_axis.axhline(0.5, color='#555555', linestyle='--', linewidth=1, label='AUC random reference')
metric_axis.set_ylim(0, 1)
metric_axis.set_ylabel('Mean score')
metric_axis.set_xlabel('')
metric_axis.set_title('Mean holdout performance, 2022–2025', fontsize=12, fontweight='bold', color='#222222')
metric_axis.tick_params(axis='x', rotation=15)
metric_axis.grid(axis='y', color='#dddddd', linewidth=0.8)
metric_axis.set_axisbelow(True)
metric_axis.legend(frameon=False, fontsize=8)
for container in metric_axis.containers:
    metric_axis.bar_label(container, fmt='%.3f', fontsize=7, padding=2)

fig.suptitle('XLK next-session direction baseline', fontsize=14, fontweight='bold', color='#222222')
fig.text(0.68, 0.02, 'Higher is better; AUC-ROC of 0.50 is approximately random ranking.', ha='center', fontsize=9, color='#444444')
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
fig.savefig(FIGURE_PATH, dpi=160, bbox_inches='tight')
print(f'Saved {FIGURE_PATH.relative_to(REPO_ROOT)}')
plt.show()

## Takeaways

### 4. Interpret the executed results

The code below turns the saved metrics into a compact, evidence-based interpretation. It compares the news model with the market-only model and the always-up reference; it does not claim causality or profitability.

In [ ]:
def mean_score(model_name, metric_name):
    values = summary_df.loc[
        (summary_df['model'] == model_name) & (summary_df['metric'] == metric_name), 'mean'
    ]
    return float(values.iloc[0]) if not values.empty else float('nan')

market_auc = mean_score('Market-only model', 'auc_roc')
news_auc = mean_score('Market + news sentiment model', 'auc_roc')
market_accuracy = mean_score('Market-only model', 'directional_accuracy')
news_accuracy = mean_score('Market + news sentiment model', 'directional_accuracy')
reference_accuracy = mean_score('Always-up reference', 'directional_accuracy')

print('XLK baseline interpretation')
print(f'- Market-only mean AUC-ROC: {market_auc:.3f}; mean directional accuracy: {market_accuracy:.1%}.')
print(f'- Market + news mean AUC-ROC: {news_auc:.3f}; mean directional accuracy: {news_accuracy:.1%}.')
print(f'- Always-up reference mean directional accuracy: {reference_accuracy:.1%}.')
print(f'- News-model AUC change versus market-only: {(news_auc - market_auc):+.3f}.')
print('These are out-of-sample descriptive results across chronological yearly holdouts, not evidence of a causal news effect or a profitable trading rule.')

### Limitations and next step

This single-asset test has fewer observations than the pooled sector experiment and may be sensitive to the chosen date window, threshold, and feature set. The next rigorous step is to compare this XLK result with the same pipeline run for the other ten sector ETFs, then test whether any apparent sentiment lift is stable across assets and years.